<a href="https://colab.research.google.com/github/StatsAI/Conv-Eval-AI/blob/main/Conv_Eval_AI_REST_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install fastapi uvicorn transformers torch pydantic nest-asyncio

import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
from transformers import pipeline
import uvicorn
import threading

In [2]:
# Initialize FastAPI app
app = FastAPI(title="Identity & Belief Evaluation API")

# Apply nest_asyncio to allow running uvicorn inside a notebook
nest_asyncio.apply()

# Load the Zero-Shot Classification model (this may take a minute)
print("Loading model... please wait.")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# --- Data Models ---

class Message(BaseModel):
    role: str  # "user" or "assistant"
    content: str
    timestamp: str

class ConversationData(BaseModel):
    conversation_id: str
    user_id: str
    messages: List[Message]
    metadata: Optional[dict] = {}

# --- Logic ---

def extract_user_beliefs(messages: List[Message]):
    # Extract only user-provided text for analysis
    user_text = " ".join([m.content for m in messages if m.role == "user"])

    if not user_text:
        return []

    # Define candidate labels for the identity signal
    candidate_labels = [
        "self-confident", "uncertain", "growth-oriented",
        "technologically savvy", "creative", "analytical",
        "community-focused", "skeptical", "risk-averse"
    ]

    # Run inference
    result = classifier(user_text, candidate_labels, multi_label=True)

    # Filter and format
    beliefs = [
        {"label": label, "confidence": round(score, 4)}
        for label, score in zip(result['labels'], result['scores'])
        if score > 0.4
    ]

    return beliefs

# --- Endpoints ---

@app.post("/evaluate-identity")
async def evaluate_conversation(data: ConversationData):
    try:
        beliefs = extract_user_beliefs(data.messages)
        return {
            "conversation_id": data.conversation_id,
            "user_id": data.user_id,
            "identity_signals": beliefs,
            "summary": "Evaluation successful" if beliefs else "No signals detected"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/")
async def root():
    return {"message": "Identity API is running!"}

Loading model... please wait.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000)

# Start the server in the background
thread = threading.Thread(target=run_server)
thread.start()

INFO:     Started server process [439]


In [6]:
import requests
import json

# Define the sample payload
payload = {
  "conversation_id": "conv-2026",
  "user_id": "pioneer-01",
  "messages": [
    {
        "role": "assistant",
        "content": "How has your experience been with the new coding tools?",
        "timestamp": "2026-03-30T09:00:00Z"
    },
    {
        "role": "user",
        "content": "I've been experimenting with asynchronous agents. I used to be intimidated by backend architecture, but now I'm building multi-agent systems from scratch. I feel much more capable.",
        "timestamp": "2026-03-30T09:01:00Z"
    }
  ],
  "metadata": {"source": "colab_test"}
}

# Send the request to the local server
response = requests.post("http://127.0.0.1:8000/evaluate-identity", json=payload)

# Print formatted results
print("--- API Response ---")
print(json.dumps(response.json(), indent=2))

INFO:     127.0.0.1:51398 - "POST /evaluate-identity HTTP/1.1" 200 OK
--- API Response ---
{
  "conversation_id": "conv-2026",
  "user_id": "pioneer-01",
  "identity_signals": [
    {
      "label": "technologically savvy",
      "confidence": 0.9826
    },
    {
      "label": "creative",
      "confidence": 0.9683
    },
    {
      "label": "uncertain",
      "confidence": 0.824
    },
    {
      "label": "growth-oriented",
      "confidence": 0.5985
    },
    {
      "label": "self-confident",
      "confidence": 0.5623
    }
  ],
  "summary": "Evaluation successful"
}
